# 18 · CP6 grounded reports and the A5 variants (T4, one session)

**T4 runtime, float16.** The estimate is about **1.5 CU** on a paid T4 (1.28 CU/h), or **0 CU** on a free-account T4. This notebook **never loads the detector**. It reads only:
- the evidence bundles written by notebook 14 (`runs/explain/p6_unified_ctx_seed42/`);
- the P6 image tar (`runs/cache/p6_images.tar`), for the image baseline and the ground truth.

**What it runs (spec §8 M6, §9 ablation 5; verifier thresholds pre-registered on 24 Nov):**
1. **Grounded reports for all 1,500 P6 images.** The model is Qwen2.5-7B-Instruct (official AWQ, pinned revision), run with vLLM, batched, at temperature 0. It reads the evidence JSON only. The rule-based verifier allows at most 2 retries. The **first pass and the final report are both kept.**
2. **No-verifier variant:** the first pass from step 1.
3. **Fixed template, no LLM:** written from the evidence record by fixed rules.
4. **LLM shown the image:** Qwen2-VL-7B-Instruct (official AWQ, pinned revision) on the 500 `P6_mllm` images. It uses the same schema, with a pixel box for each region claim.

**Metrics:**
- **GR and HR-E** on the grounded first pass.
- **HR-GT and verdict accuracy** for every variant; HR-GT is the cross-system comparison.
- **SID-Set masks with another aspect ratio** are left out of HR-GT, with the included variant reported as a sensitivity check (pre-registered 24 Nov).
- **30 reports** are saved for reading beside their evidence.

**A dropped session resumes:** reports already written are skipped.

**Results** (send to Claude): `Paper 1/runs/cp6/cp6_reports_seed42.json` and `compute_cp6.json`, plus the balance if this ran on a paid account.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings and pre-flight
import datetime, json, random, shutil, subprocess, tempfile, time
from pathlib import Path
from eviground.env import get_paths, session_info
from eviground.eval import p6cache
from eviground.explain import evidence, llm, prompts, variants, verifier
from eviground.utils import session

SEED = 42
FREE_ACCOUNT = False        # True on the free Colab account (0 CU); then CU_BALANCE_START may stay None
CU_BALANCE_START = None     # REQUIRED on a paid runtime: the balance Colab shows before Run all
EST_CU = 1.5                # about 1.2 h on a T4 at 1.28 CU/h (estimate)
VLLM_SPEC = 'vllm'          # the installed version is recorded in the result file
BATCH, VL_BATCH, N_READ = 128, 16, 30
T0 = time.time()
info = session_info(cu_per_hour=session.CU_RATES['T4'])
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp6_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    BATCH, VL_BATCH, N_READ, EST_CU = 3, 2, 2, 0.0
paths.ensure()
CK = paths.runs / 'checkpoints'
SUBSETS_FILE = CK / 'eval_subsets.json'
BUNDLES = paths.runs / 'explain' / f'p6_unified_ctx_seed{SEED}'
TAR_NAME = 'p6_images.tar'
CACHE_TAR, CACHE_MAN = paths.runs / 'cache' / TAR_NAME, paths.runs / 'cache' / 'p6_images_manifest.json'
OUT = paths.runs / 'cp6'
OUT.mkdir(parents=True, exist_ok=True)
GROUNDED_JSONL, IMAGE_JSONL = OUT / f'p6_grounded_reports_seed{SEED}.jsonl', OUT / f'p6_image_baseline_seed{SEED}.jsonl'
RESULT = OUT / f'cp6_reports_seed{SEED}.json'
HF_CACHE = str(paths.cache / 'hf') if DRY else '/content/hf_cache'

In [ ]:
# Cell 3: in the dry run, a fake P6 (images, masks, evidence bundles, cache, subsets); then the pre-flight
if DRY:
    import numpy as np
    from eviground.eval import p6data, subsets
    from eviground.eval.tiled import Item
    rng = np.random.default_rng(0)
    items, part_of = [], {}
    DRY_PROBS = {'edited': {'real': 0.05, 'generated': 0.05, 'edited': 0.9},
                 'real': {'real': 0.8, 'generated': 0.15, 'edited': 0.05},
                 'generated': {'real': 0.1, 'generated': 0.85, 'edited': 0.05}}
    for i in range(6):
        part = ['diffseg_heldout', 'sidset_tampered', 'brgen', 'real', 'generated', 'sidset_tampered'][i]
        img = (rng.random((96, 128, 3)) * 255).astype(np.uint8)
        edited = part in p6data.EDITED_PARTS
        mask = np.zeros((96, 128), bool)
        mask[20:60, 30:90] = True
        meta = {'part': part, 'label': 'edited' if edited else part}
        if i == 5:   # an aspect-mismatched SID-Set mask (flagged, as notebook 15 found 26)
            meta.update(mask_resized_from=[60, 128], mask_aspect_mismatch=True)
        uid = f'dry/{part}/{i}.png'
        items.append(Item(uid, img, mask if edited else None, None, meta)); part_of[uid] = part
        probs = DRY_PROBS['edited' if edited else part]
        mp = np.zeros((96, 128)); mp[20:60, 30:90] = 0.9 * edited
        rec, rles = evidence.evidence_record(probs, 0.6, mp, mp + rng.random((96, 128)) * 0.1, img)
        evidence.save_bundle(BUNDLES, uid, rec, rles, {'decoder': mp}, probs, 0.6)
    p6data.write_cache(items, CACHE_TAR, CACHE_MAN)
    chosen = [{'uid': u} for u in part_of]
    subsets.save({'P6': subsets._record('P6', chosen, SEED, 'dry', part_of=part_of),
                  'P6_mllm': subsets._record('P6_mllm', chosen[:4], SEED, 'dry')}, SUBSETS_FILE)
GPU = info.get('gpu_name')
if not DRY:
    assert session.gpu_type(GPU) in ('T4', 'L4'), f'needs a T4 (or an L4) GPU runtime, found {GPU}'
    assert FREE_ACCOUNT or isinstance(CU_BALANCE_START, (int, float)), 'type the balance into CU_BALANCE_START (or set FREE_ACCOUNT)'
PREFLIGHT = session.preflight(
    {'notebook': '18 CP6 reports (no detector)', 'gpu': GPU, 'grounded model': llm.QWEN25_AWQ,
     'image baseline': llm.QWEN2VL_AWQ, 'bundles': BUNDLES, 'result file': RESULT},
    paths={'evidence bundles (notebook 14)': BUNDLES, 'P6 image tar': CACHE_TAR, 'P6 tar manifest': CACHE_MAN,
           'eval_subsets.json': SUBSETS_FILE},
    est_cu=0.0 if FREE_ACCOUNT else EST_CU, cu_left=CU_BALANCE_START)
STORED = json.loads(SUBSETS_FILE.read_text())
P6, P6_MLLM = STORED['P6'], STORED['P6_mllm']
LOCAL = paths.local_data / 'p6_cache'
LOCAL_TAR = paths.local_data / TAR_NAME
shutil.copyfile(CACHE_TAR, LOCAL_TAR)
CACHE = p6cache.read_cache(LOCAL_TAR, CACHE_MAN, LOCAL)
RECORDS = {u: evidence.load_bundle(BUNDLES, u)['evidence'] for u in P6['uids']}
print(f"P6 {len(RECORDS)} evidence records, cache {CACHE['images']} images | P6_mllm {P6_MLLM['images']}")
if not DRY:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', VLLM_SPEC], check=True)
import importlib.metadata as _md
VLLM_VERSION = None if DRY else _md.version('vllm')

In [ ]:
# Cell 4: grounded reports (Qwen2.5-7B AWQ, evidence JSON only) with the verifier; first pass and final kept. Resumes.
done = set()
if GROUNDED_JSONL.is_file():
    done = {json.loads(l)['uid'] for l in GROUNDED_JSONL.read_text().splitlines() if l.strip()}
todo = [u for u in P6['uids'] if u not in done]
print(f'grounded: {len(done)} done earlier, {len(todo)} to go')
t = time.time()
if todo:
    LM = llm.MockLLM({1: 'bad_number'}) if DRY else llm.VLLMBackend(llm.QWEN25_AWQ, download_dir=HF_CACHE)
    with open(GROUNDED_JSONL, 'a') as fh:
        for i in range(0, len(todo), BATCH):
            chunk = [(u, RECORDS[u]) for u in todo[i:i + BATCH]]
            for r in verifier.run_grounded(LM, chunk):
                fh.write(json.dumps(r, default=str) + '\n')
            fh.flush()
            print(f'  {min(i + BATCH, len(todo))}/{len(todo)} | {(time.time() - t) / 60:.1f} min')
    if not DRY:
        LM.close()
GROUNDED = {r['uid']: r for r in (json.loads(l) for l in GROUNDED_JSONL.read_text().splitlines() if l.strip())}
assert set(GROUNDED) == set(P6['uids']), 'grounded reports are incomplete'
GROUNDED_MIN = round((time.time() - t) / 60, 2)
FIRST = verifier.first_pass_metrics(GROUNDED.values())
print(f"first pass: {FIRST['claims']} claims, GR {FIRST['gr']:.3f}, HR-E {FIRST['hr_e']:.3f} | rejections {FIRST['rejections_by_rule']}")

In [ ]:
# Cell 5: the image baseline (Qwen2-VL-7B AWQ sees the image; same schema, a box per region claim) on P6_mllm. Resumes.
done = set()
if IMAGE_JSONL.is_file():
    done = {json.loads(l)['uid'] for l in IMAGE_JSONL.read_text().splitlines() if l.strip()}
todo = [u for u in P6_MLLM['uids'] if u not in done]
print(f'image baseline: {len(done)} done earlier, {len(todo)} to go')
t = time.time()
if todo:
    VL = llm.MockVL() if DRY else llm.VLLMBackend(llm.QWEN2VL_AWQ, download_dir=HF_CACHE, max_model_len=8192,
                                                  limit_mm_per_prompt={'image': 1})
    with open(IMAGE_JSONL, 'a') as fh:
        for i in range(0, len(todo), VL_BATCH):
            uids = todo[i:i + VL_BATCH]
            imgs = [x['image'] for x in p6cache.cached_arrays(uids, CACHE, LOCAL)]
            for u, r in zip(uids, variants.image_reports(VL, imgs)):
                fh.write(json.dumps({'uid': u, **r}) + '\n')
            fh.flush()
    if not DRY:
        VL.close()
IMAGE = {r['uid']: r for r in (json.loads(l) for l in IMAGE_JSONL.read_text().splitlines() if l.strip())}
assert set(IMAGE) == set(P6_MLLM['uids']), 'image-baseline reports are incomplete'
IMAGE_MIN = round((time.time() - t) / 60, 2)
print(f'image baseline: {len(IMAGE)} reports ({IMAGE_MIN} min)')

In [ ]:
# Cell 6: HR-GT and verdict accuracy for every variant (SID-Set aspect-mismatched masks left out; included as sensitivity)
GT = {x['uid']: x for x in p6cache.cached_arrays(P6['uids'], CACHE, LOCAL)}
TEMPLATE = {u: variants.template_report(RECORDS[u]) for u in P6['uids']}
FLAGGED = {u for u, x in GT.items() if p6cache.is_aspect_mismatched(x['meta'], x['image'].shape[:2])}

def _rows(uids, reports, with_record=True, include_flagged=False):
    for u in uids:
        if u in FLAGGED and not include_flagged:
            continue
        x = GT[u]
        yield reports[u], x['meta']['label'], x['mask'], (RECORDS[u] if with_record else None)

VARIANTS = {
    'grounded_final': {u: GROUNDED[u]['final_report'] for u in P6['uids']},
    'no_verifier': {u: GROUNDED[u]['first_report'] for u in P6['uids']},
    'template': TEMPLATE,
}
HRGT = {}
for scope, uids in (('P6', P6['uids']), ('P6_mllm', P6_MLLM['uids'])):
    for inc in (False, True):
        key = f"{scope}{'_incl_mismatched' if inc else ''}"
        HRGT[key] = {name: verifier.hr_gt_summary(_rows(uids, reps, include_flagged=inc)) for name, reps in VARIANTS.items()}
        if scope == 'P6_mllm':
            HRGT[key]['image_baseline'] = verifier.hr_gt_summary(
                _rows(uids, {u: IMAGE[u]['report'] for u in uids}, with_record=False, include_flagged=inc))
for name, s in HRGT['P6_mllm'].items():
    print(f"P6_mllm {name:15} HR-GT {s['hr_gt']:.3f} ({s['wrong_claims']}/{s['checkable_claims']}) | verdict acc {s['verdict_accuracy']:.3f}")
FIRST_MLLM = verifier.first_pass_metrics(GROUNDED[u] for u in P6_MLLM['uids'])
TEMPLATE_PASS = sum(all(not verifier.check_claim(c, RECORDS[u]) for c in TEMPLATE[u]['claims']) for u in P6['uids'])

In [ ]:
# Cell 7: result file, 30 reports to read beside their evidence, compute record, release
READ = sorted(random.Random(SEED).sample(P6['uids'], min(N_READ, len(P6['uids']))))
with open(OUT / f'cp6_reports_to_read_seed{SEED}.json', 'w') as fh:
    json.dump([{'uid': u, 'evidence': RECORDS[u], 'first_report': GROUNDED[u]['first_report'],
                'final_report': GROUNDED[u]['final_report'], 'template': TEMPLATE[u],
                'image_baseline': IMAGE.get(u, {}).get('report')} for u in READ], fh, indent=1, default=str)
RESULT_DATA = {
    'step': 'CP6 grounded reports + A5 variants (Phase D)', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED,
    'session': info, 'vllm': VLLM_VERSION, 'models': {'grounded': llm.QWEN25_AWQ, 'image_baseline': llm.QWEN2VL_AWQ},
    'rules': {'tau_e': verifier.TAU_E, 'abs_tol': verifier.ABS_TOL, 'rel_tol': verifier.REL_TOL,
              'pct_tol': verifier.PCT_TOL, 'max_retries': verifier.MAX_RETRIES,
              'box_inside_min': verifier.BOX_INSIDE_MIN, 'temperature': 0, 'dtype': 'float16'},
    'first_pass_p6': FIRST, 'first_pass_p6_mllm': FIRST_MLLM, 'hr_gt': HRGT,
    'flagged_sidset_masks': len(FLAGGED), 'template_reports_passing_verifier': TEMPLATE_PASS,
    'retries': {k: sum(r['retries'] == k for r in GROUNDED.values()) for k in range(verifier.MAX_RETRIES + 1)},
    'dropped_claims': sum(r['dropped_claims'] or 0 for r in GROUNDED.values()),
    'verdicts_replaced_by_evidence': sum(bool(r.get('verdict_replaced')) for r in GROUNDED.values()),
    'minutes': {'grounded_this_session': GROUNDED_MIN, 'image_baseline_this_session': IMAGE_MIN},
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
RESULT.write_text(json.dumps(RESULT_DATA, indent=1, default=str))
print('saved', RESULT)
COMPUTE = session.finish(OUT, T0, gpu=GPU or 'cpu', name='compute_cp6.json', balance_start=CU_BALANCE_START,
                         extra={'git_commit': info['git_commit'], 'est_cu': EST_CU, 'free_account': FREE_ACCOUNT})
if DRY:
    assert FIRST['rejections_by_rule'].get('b', 0) >= 1          # the injected wrong number was caught
    assert TEMPLATE_PASS == len(P6['uids'])
    assert len(FLAGGED) == 1 and HRGT['P6']['template']['images'] == len(P6['uids']) - 1
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED